In [5]:
import pandas as pd
import numpy as np

In [64]:
playlist = pd.read_csv("dataset/playlist_with_segment.csv")
tracks = pd.read_csv("dataset/tracks_new.csv")

In [65]:
playlist = playlist[playlist['dataset_type'] == "train"].head(1000)
playlist['sentiment_centroid'] = playlist['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
playlist['genre_centroid'] = playlist['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

In [66]:
# Unpack the column into separate columns
playlist[['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']] = pd.DataFrame(playlist['sentiment_centroid'].tolist())

# Drop the original column if you no longer need it
playlist = playlist.drop(columns=['sentiment_centroid'])

print(playlist)

      playlist_idx  cluster dataset_type           name  \
0                1        1        train       fall '17   
1                2        3        train      HALLOWEEN   
3                4        0        train        Me Like   
4                5        0        train     Latin Trap   
5                6        1        train   My favorites   
...            ...      ...          ...            ...   
1407          1510        1        train     hype music   
1408          1511        1        train      fall 2017   
1410          1515        1        train         vibin'   
1411          1516        1        train        nutella   
1412          1518        1        train  We Found Love   

                                        name_embeddings  num_tracks  \
0     [ 1.66989997e-01 -8.01749974e-02 -2.72219986e-...          32   
1     [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...          10   
3     [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...          64   
4     [

In [67]:
# Unpack the column into separate columns
playlist[['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']] = pd.DataFrame(playlist['genre_centroid'].tolist())

# Drop the original column if you no longer need it
playlist = playlist.drop(columns=['genre_centroid'])

print(playlist)

      playlist_idx  cluster dataset_type           name  \
0                1        1        train       fall '17   
1                2        3        train      HALLOWEEN   
3                4        0        train        Me Like   
4                5        0        train     Latin Trap   
5                6        1        train   My favorites   
...            ...      ...          ...            ...   
1407          1510        1        train     hype music   
1408          1511        1        train      fall 2017   
1410          1515        1        train         vibin'   
1411          1516        1        train        nutella   
1412          1518        1        train  We Found Love   

                                        name_embeddings  num_tracks  \
0     [ 1.66989997e-01 -8.01749974e-02 -2.72219986e-...          32   
1     [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...          10   
3     [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. ...          64   
4     [

In [68]:
playlist_relevant_columns = playlist[['playlist_idx', 'track_idx_list', 'tracks_to_predict',
                                      'popularity_mean', 
                                      'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion', 'era_2000s_proportion', 'era_modern_era_proportion',
                                      'length_short_proportion', 'length_medium_proportion', 'length_long_proportion',
                                      'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
                                      'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']]

In [69]:
tracks_relevant_columns = tracks[['track_idx', 'track_popularity',
                                  'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                                  'Short', 'Medium', 'Long',
                                  "joy", "calm", "sadness", "fear", "energizing", "dreamy",
                                  "Instrumental / Ambient Sounds", "Soft Acoustic / Classical", "Orchestral / Soundtrack", "Mid-tempo Pop / Indie", "Upbeat Electronic / Dance", "Slow & Melancholic (Sad Songs)", "Experimental / Jazz Fusion", "Lo-Fi / Chill Vibes"]]

In [70]:
playlist_relevant_columns['popularity_mean'] = playlist_relevant_columns['popularity_mean'] / 100
tracks_relevant_columns['track_popularity'] = tracks_relevant_columns['track_popularity'] / 100

/var/folders/xf/v125knts0p57cdp0hhtgh8y00000gn/T/ipykernel_9297/718451119.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns['popularity_mean'] = playlist_relevant_columns['popularity_mean'] / 100
/var/folders/xf/v125knts0p57cdp0hhtgh8y00000gn/T/ipykernel_9297/718451119.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tracks_relevant_columns['track_popularity'] = tracks_relevant_columns['track_popularity'] / 100


In [71]:
import re

def convert_string_array_to_list(s):
    """Convert a string representation of an array into a list of integers."""
    if isinstance(s, str):  # Handle string case (where it's incorrectly stored)
        numbers = re.findall(r'\d+', s)  # Extract all numeric values
        return [int(x) for x in numbers]  # Convert to integers
    
    elif isinstance(s, np.ndarray):  # Handle NumPy array case
        return s.astype(int).tolist()
    
    elif isinstance(s, list):  # Handle lists with possible string numbers
        return [int(x) for x in s if str(x).isdigit()]
    
    return []  # Return empty list if the format is unexpected

# Apply function and debug output
playlist_relevant_columns['track_idx_list'] = playlist_relevant_columns['track_idx_list'].apply(lambda x: convert_string_array_to_list(x))
playlist_relevant_columns['tracks_to_predict'] = playlist_relevant_columns['tracks_to_predict'].apply(lambda x: convert_string_array_to_list(x))

# Check if the transformation worked
print(playlist_relevant_columns[['track_idx_list', 'tracks_to_predict']].head(10))  # Print first 10 rows

                                       track_idx_list  \
0   [3689, 207774, 194775, 135193, 218011, 37844, ...   
1   [160375, 131195, 164629, 147280, 193891, 17077...   
3   [244173, 210510, 9349, 224202, 147251, 35498, ...   
4   [194410, 10513, 62267, 196463, 14164, 13405, 1...   
5   [191316, 216283, 108826, 149917, 177254, 23228...   
8   [22194, 157563, 151356, 87387, 7320, 248495, 2...   
10  [68046, 122262, 157438, 27247, 176048, 90338, ...   
12  [64814, 216679, 906, 223791, 215776, 14182, 18...   
14  [241171, 93178, 247337, 86491, 4129, 89840, 24...   
15  [107175, 77255, 244728, 237699, 219025, 120223...   

                                    tracks_to_predict  
0   [218708, 242974, 165272, 9418, 221860, 229224,...  
1   [232845, 111887, 144160, 10663, 216321, 138869...  
3   [166268, 22122, 211269, 71335, 21853, 190702, ...  
4   [209088, 186942, 33032, 27612, 33426, 201531, ...  
5   [250306, 36356, 119562, 66146, 206650, 166690,...  
8   [88490, 122326, 87495, 205574, 3

/var/folders/xf/v125knts0p57cdp0hhtgh8y00000gn/T/ipykernel_9297/1335393608.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns['track_idx_list'] = playlist_relevant_columns['track_idx_list'].apply(lambda x: convert_string_array_to_list(x))
/var/folders/xf/v125knts0p57cdp0hhtgh8y00000gn/T/ipykernel_9297/1335393608.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns['tracks_to_predict'] = playlist_relevant_columns['tracks_to_predict'].apply(lambda x: conve

In [ ]:
import numpy as np
import pandas as pd

# Assuming the 'playlist_relevant_columns' and 'tracks_relevant_columns' are already defined
playlist_relevant_columns = playlist_relevant_columns.head(100)

# Function to compute Euclidean distance between a track and a playlist vector with equal contributions from each feature set
def compute_distance(track_row, playlist_row):
    # Define each feature set
    popularity_features = np.concatenate([ [track_row['track_popularity']], [playlist_row['popularity_mean']] ])
    
    era_features = np.concatenate([ 
        track_row[['Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era']].values, 
        playlist_row[['era_early_years_proportion', 'era_classic_era_proportion', 
                      'era_golden_era_proportion', 'era_2000s_proportion', 
                      'era_modern_era_proportion']].values 
    ])
    
    length_features = np.concatenate([ 
        track_row[['Short', 'Medium', 'Long']].values, 
        playlist_row[['length_short_proportion', 'length_medium_proportion', 'length_long_proportion']].values 
    ])
    
    sentiment_features = np.concatenate([ 
        track_row[['joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy']].values, 
        playlist_row[['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']].values 
    ])
    
    genre_features = np.concatenate([ 
        track_row[['Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack', 
                   'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)', 
                   'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']].values, 
        playlist_row[['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']].values 
    ])
    
    # Calculate Euclidean distances for each feature set
    popularity_distance = np.linalg.norm(popularity_features[:1] - popularity_features[1:])
    era_distance = np.linalg.norm(era_features[:5] - era_features[5:])
    length_distance = np.linalg.norm(length_features[:3] - length_features[3:])
    sentiment_distance = np.linalg.norm(sentiment_features[:6] - sentiment_features[6:])
    genre_distance = np.linalg.norm(genre_features[:8] - genre_features[8:])
    
    # Apply equal weights (1/5) to each feature set's contribution to the total distance
    total_distance = (popularity_distance + era_distance + length_distance + sentiment_distance + genre_distance) / 5
    
    return total_distance

# Initialize a list to store the distances
distances = []

# Iterate through the rows of the tracks and playlists DataFrames
for playlist_index, playlist_row in playlist_relevant_columns.iterrows():
    for track_index, track_row in tracks_relevant_columns.iterrows():
        # Compute the distance between the track and playlist
        distance = compute_distance(track_row, playlist_row)
        distances.append({
            'playlist_idx': playlist_row['playlist_idx'],
            'track_idx': track_row['track_idx'],
            'distance': distance
        })

# Convert the distances list to a DataFrame
df_distances = pd.DataFrame(distances)

# Create a function to get top 50 track recommendations
def get_top_50_recommendations(playlist_row, df_distances, track_idx_list):
    # Convert track_idx_list to a list if it's a string
    if isinstance(track_idx_list, str):
        track_idx_list = track_idx_list.split(',')  # or use ast.literal_eval() if it's a string representation of a list
    
    # Filter the distances for the given playlist
    playlist_distances = df_distances[df_distances['playlist_idx'] == playlist_row['playlist_idx']]
    
    # Sort the distances in ascending order
    playlist_distances = playlist_distances.sort_values(by='distance', ascending=True)
    
    # Get the top 50 tracks, excluding those already in 'track_idx_list'
    recommended_tracks = playlist_distances[~playlist_distances['track_idx'].isin(track_idx_list)].head(50)
    
    # Return the recommended track indices
    return recommended_tracks['track_idx'].tolist()

# Create a new column in the playlist_relevant_columns to store the top 50 recommended tracks
playlist_relevant_columns['recommendations'] = playlist_relevant_columns.apply(
    lambda row: get_top_50_recommendations(row, df_distances, row['track_idx_list']), axis=1
)

recommendation = playlist_relevant_columns[['playlist_idx', 'track_idx_list', 'tracks_to_predict', 'recommendations']]

# Check the updated DataFrame with the new column
recommendation.head()

TypeError: loop of ufunc does not support argument 0 of type float which has no callable sqrt method

In [ ]:
def compute_metrics_for_playlist(predicted_tracks, test_indices, k):
    top_k = predicted_tracks[:k]  # Consider only top K predictions

    # Hit@K
    hit = int(any(t in top_k for t in test_indices))

    # MRR and AP calculations
    precisions = []
    num_hits = 0
    mrr = 0.0

    for rank_idx, track_idx in enumerate(top_k):
        if track_idx in test_indices:
            num_hits += 1
            precision_at_k = num_hits / (rank_idx + 1)
            precisions.append(precision_at_k)
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)

    ap = np.mean(precisions) if precisions else 0.0

    return hit, mrr, ap

def evaluate_model(playlist, k):
    test_playlists = playlist
    num_playlists = len(test_playlists)

    hit_total, mrr_total, ap_total = 0, 0, 0

    for idx, row in test_playlists.iterrows():
        test_indices = row['tracks_to_predict']
        predicted_tracks = row['recommendations']

        hit, mrr, ap = compute_metrics_for_playlist(predicted_tracks, test_indices, k)
        
        hit_total += hit
        mrr_total += mrr
        ap_total += ap

    # Compute averages
    hit_ratio = hit_total / num_playlists if num_playlists > 0 else 0
    mrr_avg = mrr_total / num_playlists if num_playlists > 0 else 0
    map_avg = ap_total / num_playlists if num_playlists > 0 else 0

    return hit_ratio, mrr_avg, map_avg

# Run evaluation
hit_ratio, mrr_avg, map_avg = evaluate_model(recommendation, k=50)

# Print results
print(f"Hit@50: {hit_ratio:.4f}")
print(f"MRR: {mrr_avg:.4f}")
print(f"MAP@50: {map_avg:.4f}")

Hit@50: 0.0000
MRR: 0.0000
MAP@50: 0.0000
